In [ ]:
# ============================================================
# PROJECT 7: AUTO MPG
# LINEAR REGRESSION, FEATURE SCALING & ENCODING
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("File:", file_name)


# ------------------------------------------------------------
# 3. BASIC DATA EXPLORATION
# ------------------------------------------------------------

print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== LAST 5 ROWS ==========")
display(df.tail())

print("\n========== DATASET SHAPE ==========")
print(df.shape)

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFORMATION ==========")
df.info()

print("\n========== STATISTICAL SUMMARY ==========")
display(df.describe(include="all").T)

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 4. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 5. HANDLE '?' VALUES
# ------------------------------------------------------------

df = df.replace("?", np.nan)

print("\nMissing values after replacing '?':")
print(df.isnull().sum())


# ------------------------------------------------------------
# 6. CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_candidates = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year"
]

for column in numeric_candidates:

    if column in df.columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# 7. REMOVE DUPLICATES
# ------------------------------------------------------------

before_duplicates = df.shape[0]

df = df.drop_duplicates()

after_duplicates = df.shape[0]

print("\nDuplicates removed:", before_duplicates - after_duplicates)
print("New dataset shape:", df.shape)


# ------------------------------------------------------------
# 8. CHECK REQUIRED TARGET
# ------------------------------------------------------------

TARGET = "mpg"

if TARGET not in df.columns:

    raise ValueError(
        "The target column 'mpg' was not found. "
        "Please check your dataset column names."
    )


# ------------------------------------------------------------
# 9. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

print("\n========== MPG SUMMARY ==========")

print("Mean MPG   :", df[TARGET].mean())
print("Median MPG :", df[TARGET].median())
print("Minimum MPG:", df[TARGET].min())
print("Maximum MPG:", df[TARGET].max())


# ------------------------------------------------------------
# 10. MPG DISTRIBUTION
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.histplot(
    df[TARGET].dropna(),
    kde=True
)

plt.title("Distribution of MPG")
plt.xlabel("Miles Per Gallon (MPG)")
plt.ylabel("Frequency")

plt.show()


# ------------------------------------------------------------
# 11. CORRELATION HEATMAP
# ------------------------------------------------------------

numeric_df = df.select_dtypes(
    include=np.number
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()


# ------------------------------------------------------------
# 12. MPG VS HORSEPOWER
# ------------------------------------------------------------

if "horsepower" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.scatterplot(
        data=df,
        x="horsepower",
        y="mpg"
    )

    plt.title("MPG vs Horsepower")
    plt.xlabel("Horsepower")
    plt.ylabel("MPG")

    plt.show()


# ------------------------------------------------------------
# 13. MPG VS WEIGHT
# ------------------------------------------------------------

if "weight" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.scatterplot(
        data=df,
        x="weight",
        y="mpg"
    )

    plt.title("MPG vs Vehicle Weight")
    plt.xlabel("Weight")
    plt.ylabel("MPG")

    plt.show()


# ------------------------------------------------------------
# 14. MPG VS DISPLACEMENT
# ------------------------------------------------------------

if "displacement" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.scatterplot(
        data=df,
        x="displacement",
        y="mpg"
    )

    plt.title("MPG vs Displacement")
    plt.xlabel("Displacement")
    plt.ylabel("MPG")

    plt.show()


# ------------------------------------------------------------
# 15. MPG BY CYLINDERS
# ------------------------------------------------------------

if "cylinders" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="cylinders",
        y="mpg"
    )

    plt.title("MPG by Number of Cylinders")
    plt.xlabel("Cylinders")
    plt.ylabel("MPG")

    plt.show()


# ------------------------------------------------------------
# 16. MPG BY MODEL YEAR
# ------------------------------------------------------------

if "model_year" in df.columns:

    plt.figure(figsize=(10, 5))

    sns.lineplot(
        data=df,
        x="model_year",
        y="mpg",
        marker="o"
    )

    plt.title("Average MPG by Model Year")
    plt.xlabel("Model Year")
    plt.ylabel("MPG")

    plt.show()


# ------------------------------------------------------------
# 17. SELECT FEATURES
# ------------------------------------------------------------

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget:")
print(TARGET)


# ------------------------------------------------------------
# 18. IDENTIFY NUMERIC & CATEGORICAL FEATURES
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


print("\n========== NUMERIC FEATURES ==========")
print(numeric_features)

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)


# ------------------------------------------------------------
# 19. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n========== TRAIN-TEST SPLIT ==========")

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


# ------------------------------------------------------------
# 20. NUMERIC PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 21. CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 22. COMBINE PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 23. CREATE LINEAR REGRESSION MODEL
# ------------------------------------------------------------

model = LinearRegression()


# ------------------------------------------------------------
# 24. COMPLETE PIPELINE
# ------------------------------------------------------------

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ]
)


# ------------------------------------------------------------
# 25. TRAIN MODEL
# ------------------------------------------------------------

print("\nTraining Linear Regression model...")

pipeline.fit(
    X_train,
    y_train
)

print("Model training completed!")


# ------------------------------------------------------------
# 26. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = pipeline.predict(
    X_test
)


# ------------------------------------------------------------
# 27. MODEL EVALUATION
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)


print("\n========== MODEL PERFORMANCE ==========")

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ------------------------------------------------------------
# 28. ACTUAL VS PREDICTED
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual MPG": y_test.values,
    "Predicted MPG": y_pred
})

print("\n========== ACTUAL VS PREDICTED ==========")

display(
    comparison.head(20)
)


# ------------------------------------------------------------
# 29. ACTUAL VS PREDICTED PLOT
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.title("Actual vs Predicted MPG")
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")

plt.show()


# ------------------------------------------------------------
# 30. RESIDUAL ANALYSIS
# ------------------------------------------------------------

residuals = y_test - y_pred

plt.figure(figsize=(8, 5))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.title("Residual Plot")
plt.xlabel("Predicted MPG")
plt.ylabel("Residuals")

plt.show()


# ------------------------------------------------------------
# 31. RESIDUAL DISTRIBUTION
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.histplot(
    residuals,
    kde=True
)

plt.title("Distribution of Residuals")
plt.xlabel("Residual")
plt.ylabel("Frequency")

plt.show()


# ------------------------------------------------------------
# 32. GET PROCESSED FEATURE NAMES
# ------------------------------------------------------------

feature_names = (
    pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("\nNumber of processed features:")
print(len(feature_names))


# ------------------------------------------------------------
# 33. GET LINEAR REGRESSION COEFFICIENTS
# ------------------------------------------------------------

coefficients = (
    pipeline
    .named_steps["model"]
    .coef_
)

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coefficient_df["AbsoluteCoefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    by="AbsoluteCoefficient",
    ascending=False
)


print("\n========== MODEL COEFFICIENTS ==========")

display(
    coefficient_df.head(20)
)


# ------------------------------------------------------------
# 34. PLOT TOP COEFFICIENTS
# ------------------------------------------------------------

top_coefficients = (
    coefficient_df
    .head(15)
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_coefficients,
    x="Coefficient",
    y="Feature"
)

plt.title(
    "Top Linear Regression Coefficients"
)

plt.xlabel("Coefficient")
plt.ylabel("Feature")

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 35. MODEL INTERCEPT
# ------------------------------------------------------------

intercept = (
    pipeline
    .named_steps["model"]
    .intercept_
)

print("\nModel Intercept:")
print(intercept)


# ------------------------------------------------------------
# 36. PREDICT A NEW CAR
# ------------------------------------------------------------

# Use the first test row as an example
# so the code automatically matches your dataset structure.

new_car = X_test.iloc[[0]].copy()

print("\n========== SAMPLE CAR ==========")

display(
    new_car
)


new_prediction = pipeline.predict(
    new_car
)

print(
    "\nPredicted MPG for sample car:",
    round(new_prediction[0], 2)
)


# ------------------------------------------------------------
# 37. SAVE PREDICTIONS
# ------------------------------------------------------------

prediction_results = X_test.copy()

prediction_results["Actual_MPG"] = y_test.values

prediction_results["Predicted_MPG"] = y_pred

prediction_results["Residual"] = (
    prediction_results["Actual_MPG"]
    - prediction_results["Predicted_MPG"]
)


prediction_file = "Auto_MPG_Linear_Regression_Predictions.csv"

prediction_results.to_csv(
    prediction_file,
    index=False
)

print("\nPrediction file saved as:")
print(prediction_file)


# ------------------------------------------------------------
# 38. DOWNLOAD RESULTS
# ------------------------------------------------------------

files.download(
    prediction_file
)


# ------------------------------------------------------------
# 39. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("PROJECT 7 - FINAL SUMMARY")
print("=" * 60)

print("Dataset shape:", df.shape)

print("\nTarget variable:", TARGET)

print("\nModel:")
print("Linear Regression")

print("\nPreprocessing:")
print("- Missing value imputation")
print("- StandardScaler for numerical features")
print("- OneHotEncoder for categorical features")

print("\nEvaluation Metrics:")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print("\nThe model predicts vehicle fuel efficiency")
print("using Linear Regression with feature scaling")
print("and categorical feature encoding.")

print("=" * 60)